In [5]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
UTILS = str(ROOT / "src" / "utils")
if UTILS not in sys.path:
    sys.path.insert(0, UTILS)

import polars as pl
from pgn import read_games

PGN_PATH = "../../Data/lichess_db_standard_rated_2026-09.pgn.zst"
RAW_PATH = "../../Data/blitz_180_raw.parquet"

if not Path(RAW_PATH).exists():
    df = read_games(PGN_PATH, n_read=10_000, time_controls={"180+0", "180+2"})
    df.write_parquet(RAW_PATH)

df = pl.read_parquet(RAW_PATH)

In [6]:
import chess
import chess.svg
import ipywidgets as widgets
from IPython.display import display

sans = [move["san"] for move in df["moves"][0]]

boards = [chess.Board()]
for san in sans:
    b = boards[-1].copy()
    b.push_san(san)
    boards.append(b)

ply = widgets.BoundedIntText(value=0, min=0, max=len(sans), description="Coup :")
prev_btn = widgets.Button(description="< Precedent")
next_btn = widgets.Button(description="Suivant >")
board_html = widgets.HTML()

def render(_=None):
    i = ply.value
    last = boards[i].move_stack[-1] if boards[i].move_stack else None
    svg = chess.svg.board(boards[i], size=400, lastmove=last)
    board_html.value = svg

def go_prev(_):
    ply.value = max(0, ply.value - 1)

def go_next(_):
    ply.value = min(len(sans), ply.value + 1)

prev_btn.on_click(go_prev)
next_btn.on_click(go_next)
ply.observe(render, names="value")

render()
display(widgets.VBox([widgets.HBox([prev_btn, next_btn, ply]), board_html]))